# Mini Projeto de Data Science — Titanic

## Problema
Queremos investigar quais fatores parecem estar mais associados à sobrevivência no Titanic.

## Pergunta principal
Quais características dos passageiros parecem influenciar a chance de sobrevivência?

## Objetivos
- Carregar e inspecionar um dataset real
- Fazer limpeza básica de dados
- Explorar variáveis numéricas e categóricas
- Criar visualizações
- Formular hipóteses
- Interpretar resultados

## Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind, chi2_contingency

sns.set_style("whitegrid")

## Carregando dataset

In [ ]:
df = sns.load_dataset("titanic")
df.head()

## Entendendo a estrutura

In [ ]:
df.info()

In [ ]:
df.describe(include="all")

## Perguntas iniciais

Antes de começar a análise, tente responder:

1. Homens e mulheres tiveram a mesma chance de sobreviver?
2. A classe do passageiro pode ter influenciado?
3. Crianças sobreviveram mais?
4. O preço da passagem pode indicar algo?
5. Estar sozinho ou acompanhado pode fazer diferença?

### Parte 1 — Limpeza e preparação

#### Valores ausentes

In [ ]:
df.isnull().sum()

Explique:

* age possui valores ausentes
* deck tem muitos ausentes
* embark_town e embarked também

#### Selecionando colunas úteis

In [ ]:
df = df[['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'class', 'who', 'alone']]
df.head()

#### Tratamento simples dos ausentes

In [ ]:
df['age'] = df['age'].fillna(df['age'].median())
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])

#### Conferindo novamente

In [ ]:
df.isnull().sum()

### Parte 2 — Entendendo a variável alvo

#### Taxa de sobrevivência geral

In [ ]:
df['survived'].mean()

Explique:

> como survived é 0/1, a média representa a proporção de sobreviventes

#### Gráfico de sobrevivência

In [ ]:
df['survived'].value_counts().plot.bar()
plt.title("Sobreviveu (1) vs Não Sobreviveu (0)")
plt.xlabel("Sobreviveu")
plt.ylabel("Quantidade")
plt.show()

### Parte 3 — Análise exploratória

#### Sobrevivência por sexo

In [ ]:
df.groupby('sex')['survived'].mean()

#### Gráfico por sexo

In [ ]:
sns.barplot(data=df, x='sex', y='survived')
plt.title("Taxa de Sobrevivência por Sexo")
plt.ylabel("Proporção de Sobreviventes")
plt.show()

Pergunta para a turma:

> O sexo parece influenciar a sobrevivência?

#### Sobrevivência por classe

In [ ]:
df.groupby('class')['survived'].mean()

#### Gráfico por classe

In [ ]:
sns.barplot(data=df, x='class', y='survived', order=['First', 'Second', 'Third'])
plt.title("Taxa de Sobrevivência por Classe")
plt.ylabel("Proporção de Sobreviventes")
plt.show()

Pergunta:

> Passageiros de classes mais altas tiveram vantagem?

#### Idade e sobrevivência

In [ ]:
plt.figure(figsize=(8,4))
sns.histplot(data=df, x='age', hue='survived', bins=30, kde=True, multiple='stack')
plt.title("Distribuição de Idade por Sobrevivência")
plt.show()

#### Boxplot da idade

In [ ]:
sns.boxplot(data=df, x='survived', y='age')
plt.title("Idade por Sobrevivência")
plt.show()

Pergunta:

> A idade parece diferenciar os grupos?

#### Tarifa e sobrevivência

In [ ]:
sns.boxplot(data=df, x='survived', y='fare')
plt.title("Tarifa por Sobrevivência")
plt.show()

Explique:

* fare costuma ter outliers
* boxplot ajuda muito aqui

#### Passageiro sozinho ou acompanhado

In [ ]:
df.groupby('alone')['survived'].mean()

#### Gráfico sozinho vs acompanhado

In [ ]:
sns.barplot(data=df, x='alone', y='survived')
plt.title("Sobrevivência: Sozinho vs Acompanhado")
plt.ylabel("Proporção de Sobreviventes")
plt.show()

### Parte 4 — Engenharia de atributo simples

#### Criando tamanho da família

In [ ]:
df['family_size'] = df['sibsp'] + df['parch'] + 1
df[['sibsp', 'parch', 'family_size']].head()

#### Sobrevivência por tamanho da família

In [ ]:
df.groupby('family_size')['survived'].mean()

In [ ]:
plt.figure(figsize=(8,4))
sns.barplot(data=df, x='family_size', y='survived')
plt.title("Sobrevivência por Tamanho da Família")
plt.ylabel("Proporção de Sobreviventes")
plt.show()

### Parte 5 — Correlações e relações

#### 5 — Correlações e relações

In [ ]:
corr = df[['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare', 'family_size']].corr()

plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title("Matriz de Correlação")
plt.show()

Explique:
* correlação ajuda, mas não explica tudo
* para categóricas, usamos outras abordagens

### Parte 6 — Hipóteses estatísticas simples

#### Hipótese 1 — Idade dos sobreviventes vs não sobreviventes

In [ ]:
idades_sobreviveu = df[df['survived'] == 1]['age']
idades_nao_sobreviveu = df[df['survived'] == 0]['age']

t_stat, p_value = ttest_ind(idades_sobreviveu, idades_nao_sobreviveu, equal_var=False)

print("t =", t_stat)
print("p-value =", p_value)

Interpretação em sala:
* H0: as médias de idade são iguais
* H1: as médias são diferentes

#### Hipótese 2 — Sexo e sobrevivência (qui-quadrado)

In [ ]:
tabela_sexo = pd.crosstab(df['sex'], df['survived'])
tabela_sexo

In [ ]:
chi2, p, dof, expected = chi2_contingency(tabela_sexo)

print("Qui-quadrado:", chi2)
print("p-value:", p)

Interpretação:
* H0: sexo e sobrevivência são independentes
* H1: existe associação entre sexo e sobrevivência

#### Hipótese 3 — Classe e sobrevivência

In [ ]:
tabela_classe = pd.crosstab(df['class'], df['survived'])
tabela_classe

In [ ]:
chi2, p, dof, expected = chi2_contingency(tabela_classe)

print("Qui-quadrado:", chi2)
print("p-value:", p)

## Conclusão

Com base na análise exploratória e nos testes realizados, observamos que:

- **sexo** parece estar fortemente associado à sobrevivência
- **classe** também parece influenciar bastante
- **idade** pode ter algum efeito, mas precisa ser interpretada com cuidado
- **tarifa** parece indicar diferenças socioeconômicas relevantes
- variáveis relacionadas à **família** também podem ser úteis

## Observação importante
Esses resultados mostram **associações**, não necessariamente causalidade.

# Exercícios

## Exercício 1
Calcule a taxa de sobrevivência por `who` e interprete.

## Exercício 2
Crie um gráfico comparando a sobrevivência por `embarked`.

## Exercício 3
Verifique se a tarifa (`fare`) difere entre as classes.

## Exercício 4
Escreva 3 conclusões de negócio ou produto com base nos dados.

## Exercício 5
Responda:
Se você fosse criar um modelo preditivo, quais variáveis escolheria primeiro?

# Fechamento

Neste mini projeto passamos por etapas muito comuns em Data Science:

1. definição do problema
2. carregamento dos dados
3. limpeza
4. análise exploratória
5. formulação de hipóteses
6. teste estatístico
7. conclusão

Esse fluxo é a base de muitos projetos reais de análise de dados.